# ResNet-18 from Scratch (PyTorch)

Implementation of a residual network, following [D2L §8.6 Residual Networks (ResNet)](https://d2l.ai/chapter_convolutional-modern/resnet.html) and the original paper [He et al., 2016 — *Deep Residual Learning for Image Recognition*](https://arxiv.org/abs/1512.03385).

**Contents**
1. The residual block (`BasicBlock`)
2. Assembling ResNet-18
3. Sanity checks: output shapes and parameter count
4. Training on Fashion-MNIST

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lee2716/Dive_into_Deep_Learning_Code_Implementation/blob/main/my_resnet.ipynb)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. The Residual Block

Instead of asking a stack of layers to learn a mapping $H(x)$ directly, a residual block learns the **residual** $F(x) = H(x) - x$ and outputs

$$y = \mathrm{ReLU}\big(F(x) + x\big)$$

Why this helps:
- If the best thing a block can do is nothing, it only has to push $F(x) \to 0$, which is much easier than learning an identity map through several non-linear layers.
- The shortcut gives gradients a direct path backwards, so very deep networks stay trainable.

`BasicBlock` = two 3×3 conv layers, each followed by BatchNorm. When the block changes the spatial size (`stride=2`) or the number of channels, the shortcut `x` no longer matches $F(x)$, so a 1×1 conv (`downsample`) reshapes it.

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1  # output channels = out_channels * expansion (4 for Bottleneck blocks in ResNet-50+)

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)

        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out += identity          # the residual connection
        return F.relu(out)

Quick check: a block that keeps the shape, and a block that halves the resolution and doubles the channels.

In [ ]:
x = torch.randn(4, 64, 56, 56)

same = BasicBlock(64, 64)
print(same(x).shape)   # -> (4, 64, 56, 56)

down = BasicBlock(64, 128, stride=2,
                  downsample=nn.Sequential(nn.Conv2d(64, 128, kernel_size=1, stride=2, bias=False),
                                           nn.BatchNorm2d(128)))
print(down(x).shape)   # -> (4, 128, 28, 28)

## 2. Assembling ResNet-18

| Stage | Output channels | Blocks | Spatial size (224 input) |
|-------|-----------------|--------|--------------------------|
| stem: 7×7 conv, stride 2 + 3×3 max-pool | 64 | – | 56×56 |
| `layer1` | 64 | 2 | 56×56 |
| `layer2` | 128 | 2 | 28×28 |
| `layer3` | 256 | 2 | 14×14 |
| `layer4` | 512 | 2 | 7×7 |
| global avg-pool + FC | `num_classes` | – | 1×1 |

1 stem conv + 4 stages × 2 blocks × 2 convs + 1 FC = **18 layers** with weights.

`_make_layer` builds one stage: only the first block of a stage may change stride/channels (and so needs `downsample`); the rest keep the shape.

`in_channels` is configurable so the same model works for RGB (3) and grayscale (1) images.

In [ ]:
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000, in_channels=3):
        super().__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion),
            )

        layers = [block(self.in_channels, out_channels, stride, downsample)]
        self.in_channels = out_channels * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


def resnet18(num_classes=1000, in_channels=3):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes, in_channels)

## 3. Sanity Checks

Trace a 224×224 image through every stage. The parameter count should be about **11.7 M**, matching `torchvision.models.resnet18`.

In [ ]:
net = resnet18()
X = torch.randn(1, 3, 224, 224)
for name, layer in net.named_children():
    X = layer(X)
    if name == 'avgpool':
        X = torch.flatten(X, 1)
    print(f'{name:8s} output shape: {tuple(X.shape)}')

n_params = sum(p.numel() for p in net.parameters())
print(f'\nTotal parameters: {n_params / 1e6:.2f} M')

## 4. Training on Fashion-MNIST

As in D2L, images are resized to 96×96 to keep training fast. A GPU is strongly recommended (in Colab: *Runtime → Change runtime type → GPU*).

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

batch_size, lr, num_epochs = 128, 0.05, 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

trans = transforms.Compose([transforms.Resize(96), transforms.ToTensor()])
train_set = datasets.FashionMNIST(root='./data', train=True, transform=trans, download=True)
test_set = datasets.FashionMNIST(root='./data', train=False, transform=trans, download=True)
train_iter = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
test_iter = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
def evaluate_accuracy(net, data_iter):
    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            correct += (net(X).argmax(dim=1) == y).sum().item()
            total += y.numel()
    return correct / total


def train(net, train_iter, test_iter, num_epochs, lr):
    net.to(device)
    optimizer = torch.optim.SGD(net.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}
    for epoch in range(num_epochs):
        net.train()
        loss_sum, correct, total = 0.0, 0, 0
        for X, y in train_iter:
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            loss = loss_fn(y_hat, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * y.numel()
            correct += (y_hat.argmax(dim=1) == y).sum().item()
            total += y.numel()
        test_acc = evaluate_accuracy(net, test_iter)
        history['train_loss'].append(loss_sum / total)
        history['train_acc'].append(correct / total)
        history['test_acc'].append(test_acc)
        print(f'epoch {epoch + 1:2d}: loss {loss_sum / total:.3f}, '
              f'train acc {correct / total:.3f}, test acc {test_acc:.3f}')
    return history

In [ ]:
model = resnet18(num_classes=10, in_channels=1)
history = train(model, train_iter, test_iter, num_epochs, lr)

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, num_epochs + 1)
plt.plot(epochs, history['train_loss'], label='train loss')
plt.plot(epochs, history['train_acc'], label='train acc')
plt.plot(epochs, history['test_acc'], label='test acc')
plt.xlabel('epoch')
plt.legend()
plt.grid(True)
plt.show()

## Exercises

1. Remove the shortcut (`out += identity`) and compare training curves. What happens as you add more blocks?
2. Implement the `Bottleneck` block (1×1 → 3×3 → 1×1, `expansion = 4`) and build ResNet-50 with `layers=[3, 4, 6, 3]`.
3. For 28×28 inputs, replace the 7×7 stem with a single 3×3 conv (stride 1) and drop the max-pool. Does accuracy improve without resizing to 96×96?